# NN Guided QAOA and Hadamard matrix Discovery

Let's install the necessary packages:

In [80]:
import numpy as np
from itertools import product

Now
Generate all possible Hadamard matrices

In [81]:
matrices = []

for bits in product([-1, 1], repeat=4):
    H = np.array(bits).reshape(2, 2)
    matrices.append(H)

In [82]:
# matrices                  # To look at all the matrices
print(len(matrices))

16


In [83]:
H

array([[1, 1],
       [1, 1]])

We define a function to generate Hadamard matrices.

In [84]:
def is_hadamard(H):
    n = H.shape[0]
    return np.array_equal(H @ H.T, n * np.eye(n))

Then

In [85]:
X = []
y = []

for H in matrices:

    X.append(H.flatten())

    if is_hadamard(H):
        y.append(1)
    else:
        y.append(0)

X = np.array(X)
y = np.array(y)

Here `X` contains all the contains our candidate matrices and `y` contains the corresponding labels.


for
y = 0 --> invalid Hadamard matrix


y = 1 --> valid Hadamard matrix

In [86]:
X

array([[-1, -1, -1, -1],
       [-1, -1, -1,  1],
       [-1, -1,  1, -1],
       [-1, -1,  1,  1],
       [-1,  1, -1, -1],
       [-1,  1, -1,  1],
       [-1,  1,  1, -1],
       [-1,  1,  1,  1],
       [ 1, -1, -1, -1],
       [ 1, -1, -1,  1],
       [ 1, -1,  1, -1],
       [ 1, -1,  1,  1],
       [ 1,  1, -1, -1],
       [ 1,  1, -1,  1],
       [ 1,  1,  1, -1],
       [ 1,  1,  1,  1]])

In [87]:
print(X.shape)
print(y.shape)

(16, 4)
(16,)


In [88]:
y

array([0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0])

### Let's inspect the dataset

In [89]:
print("Input shape:", X.shape)
print("Label shape:", y.shape)
print("Number of valid matrices:", np.sum(y))           # sum of '1's in y
print("Number of invalid matrices:", np.sum(1 - y))

Input shape: (16, 4)
Label shape: (16,)
Number of valid matrices: 8
Number of invalid matrices: 8


## **Now let's build the Actual Neural Network**

`First Layer`
we need connections from 4 inputs to 2 hidden neurons
Therefore we need 8 weights 
$$W_1 \in \mathbb{R}^{2 \times 4}$$

In Python
```python
W1 = np.random.randn(2, 4) * 0.1
```

why (2, 4) and not (4, 2) ?
Because we need 4 inputs, so we need 4 inputs for 2 hidden neurons.

In [90]:
W1 = np.random.randn(2, 4) * 0.1     # 2 rows and 4 columns in W1 randomly
# We are multiplying with 0.1 to make the weights small

print(W1)
print(W1.shape)

[[-0.14088592 -0.13772492 -0.08274569 -0.08276267]
 [ 0.09297049 -0.2965047  -0.00384753 -0.02165385]]
(2, 4)


`Hidden Layer bias`
We have two hidden neurons, we need two biases

```python
b1 = np.zeros(2)
```
Therefore 
$$b_1 \in \mathbb{R}^{2}$$

In [91]:
b1 = np.zeros(2)                     # bias, 2 rows in b1 with only zeros

print(b1)
print(b1.shape)

[0. 0.]
(2,)


`Second Layer`
We need connections from 2 hidden neurons to 1 output neuron
Therefore we need 2 weights
$$W_2 \in \mathbb{R}^{1 \times 2}$$

```python
W2 = np.random.randn(1, 2) * 0.1
```

In [92]:
W2 = np.random.randn(1, 2) * 0.1            # 1 row and 2 columns in W2 randomly

# We are multiplying with 0.1 to make the weights small

print(W2)
print(W2.shape)

[[ 0.0653182  -0.14656621]]
(1, 2)






`Output Layer bias` 
One output neuron
$$b_2 \in \mathbb{R}^{1}$$

```python
b2 = np.zeros(1)
```



In [93]:
b2 = np.zeros(1)                     # bias, 1 row in b2 with only zeros

print(b2)
print(b2.shape)

[0.]
(1,)


## **Activation Function**
For the hidden layer, let's use the sigmoid function first because it is easy to understand
$$\sigma(z)= \frac{1}{1+e^{-z}}$$

In [94]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [95]:
# For exmple
print(sigmoid(0))


# or 
print(sigmoid(10))

# or
print(sigmoid(-10))

0.5
0.9999546021312976
4.5397868702434395e-05


## **Forward Propagation**
Now we implement the network.

The first calculation is: 
$$Z_1 = XW_1^T + b_1$$

In [96]:
Z1 = X @ W1.T + b1

print(Z1)
print(Z1.shape)

[[ 0.4441192   0.22903559]
 [ 0.27859386  0.18572789]
 [ 0.27862782  0.22134052]
 [ 0.11310248  0.17803283]
 [ 0.16866936 -0.36397381]
 [ 0.00314402 -0.40728151]
 [ 0.00317798 -0.37166887]
 [-0.16234736 -0.41497657]
 [ 0.16234736  0.41497657]
 [-0.00317798  0.37166887]
 [-0.00314402  0.40728151]
 [-0.16866936  0.36397381]
 [-0.11310248 -0.17803283]
 [-0.27862782 -0.22134052]
 [-0.27859386 -0.18572789]
 [-0.4441192  -0.22903559]]
(16, 2)


In [97]:
# Then

A1 = sigmoid(Z1)

print(A1)
print(A1.shape)

[[0.60924012 0.5570099 ]
 [0.56920146 0.54629896]
 [0.56920978 0.55511032]
 [0.52824552 0.54439102]
 [0.54206765 0.40999796]
 [0.500786   0.39956414]
 [0.5007945  0.40813782]
 [0.45950207 0.39771944]
 [0.54049793 0.60228056]
 [0.4992055  0.59186218]
 [0.499214   0.60043586]
 [0.45793235 0.59000204]
 [0.47175448 0.45560898]
 [0.43079022 0.44488968]
 [0.43079854 0.45370104]
 [0.39075988 0.4429901 ]]
(16, 2)


THis is the activation of our two hidden neurons.

So $$A_1 = \sigma(Z_1)$$


The second calculation is:
$$Z_2 = A_1W_2^T + b_2$$

In [98]:
Z2 = A1 @ W2.T + b2

print(Z2)
print(Z2.shape)

[[-0.04184436]
 [-0.04288976]
 [-0.04418066]
 [-0.04528528]
 [-0.02468497]
 [-0.02585216]
 [-0.02710822]
 [-0.02827838]
 [-0.05296963]
 [-0.05413979]
 [-0.05539585]
 [-0.05656305]
 [-0.03596273]
 [-0.03706735]
 [-0.03835826]
 [-0.03940365]]
(16, 1)


In [99]:
# Finally
y_pred = sigmoid(Z2)

print(y_pred)
print(y_pred.shape)

[[0.48954043]
 [0.4892792 ]
 [0.48895663]
 [0.48868061]
 [0.49382907]
 [0.49353732]
 [0.49322336]
 [0.49293088]
 [0.48676069]
 [0.48646836]
 [0.48615458]
 [0.48586301]
 [0.49101029]
 [0.49073422]
 [0.49041161]
 [0.49015036]]
(16, 1)


Now 
$$\hat{y} = \sigma(Z_2)$$
is our predicted probability of the Hadamard matrix being valid.

> NOTE: Here $\sigma(z)$ is the sigmoid function.

### But our NN is currently random
At this point, `y_pred` is a random output and meaningless. Because $W_1, b_1, W_2, b_2$ are random. 

The NN has not learned anything yet. We need to train the NN. 

So now we need **loss + backpropagation + gradient descent**, and this is where the actual training starts.

## **Loss Function**
Because this is binary classification (Hadamard Matrix or not), we use the binary cross entropy loss function.
$$L = -\frac{1}{N}\sum_{i=1}^N y_i\log(\hat{y_i}) + (1-y)\log(1-\hat{y_i})$$

The objective is to adjust the parameters ($W_1, b_1, W_2, b_2$) to minimize the loss.

In [100]:
def binary_cross_entropy(y, y_pred):
    epsilon = 1e-8

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)     # epsilon  is just to avoid log(0) because log(0) is undefined

    return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

## First Forward Pass
Now run:

In [106]:
loss = binary_cross_entropy(y.reshape(-1, 1), y_pred)   # y.reshape(-1, 1) is to make 'y' a 2D array

print("Initial Loss",loss)

Initial Loss 0.6933667050351213


At this point, the Hadamard mathematics is still being handleded by our `is_hadamard()` function. 

> Can we remove the need to explicitly enumerate all v alid solutions and use the NN to **guide the search towards a valid Hadamard matrix?**

The NN has not learned anything yet. The loss is high.

## **Backpropagation**
$$\frac{\partial L}{\partial W_2}, \frac{\partial L}{\partial b_2}, \frac{\partial L}{\partial W_1}, \frac{\partial Z_1}{\partial b_1}$$
We will derive it rather than hiding it inside a library. 

The chain will be like this:
Loss $\Rightarrow$ Output Layer $\Rightarrow$ Hidden Layer $\Rightarrow$ $W_1, b_1$

Then we will update the weights and biases.
$$W \leftarrow W - \eta \frac{\partial L}{\partial W}$$

$$b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

where $\eta$ is the learning rate. 
This will give us a complete neural network written by ourselves.

For sigmoid output combined with binary cross entropy, the derivative simplifies to:
$$dZ_2= \frac{\hat{y}-y}{N}$$
This tells us how much the output error changes the output neuron's input $Z_2$.

In [108]:
dZ2 =(y_pred - y.reshape(-1, 1))/len(y)

# y_pred predicted probabilities, y.reshape(-1, 1) convert the labels into a column vector

print(dZ2)
print(dZ2.shape)

[[ 0.03059628]
 [-0.03192005]
 [-0.03194021]
 [ 0.03054254]
 [-0.03163568]
 [ 0.03084608]
 [ 0.03082646]
 [-0.03169182]
 [-0.03207746]
 [ 0.03040427]
 [ 0.03038466]
 [-0.03213356]
 [ 0.03068814]
 [-0.03182911]
 [-0.03184927]
 [ 0.0306344 ]]
(16, 1)


### Calculate the output weights and bias gradients
The output depends on the hidden-layer activation and the output weights.

$$\boxed{dW_2 = dZ_2^T A_1}$$
$$\boxed{db_2 = \sum_{i=1}^N dZ_{2,i}}$$

In [110]:
dW2 = dZ2.T @ A1
db2 = np.sum(dZ2, axis=0)

# These are the gradients that tells us how to adjust the weights and biases to reduce the loss.

print(dW2)
print(dW2.shape)

[[-0.00509201 -0.00526458]]
(1, 2)


### Send the Error Backward to the Hidden Layer
The hidden neurons also contributed to the output error. We therefore propagate that error backward thought the output weights.

$$\boxed{dA_1 = dZ_2 W_2}$$

The derivative of the activation function (sigmoid) is:
$$\sigma'(z) = \sigma(z)(1-\sigma(z))$$

Therefore, 
$$\boxed{dZ_1 = dA_1 \odot A_1(1-A_1)}$$

The symbol $\odot$ is the element-by-element multiplication.

In [111]:
dA1 = dZ2 @ W2
dZ1 = dA1 * A1 * (1 - A1)

print(dZ1)
print(dZ1.shape)

[[ 0.00047577 -0.00110652]
 [-0.00051126  0.00115957]
 [-0.00051158  0.00115612]
 [ 0.00049715 -0.0011103 ]
 [-0.00051294  0.00112162]
 [ 0.0005037  -0.00108464]
 [ 0.00050338 -0.0010914 ]
 [-0.00051412  0.00111265]
 [-0.00052037  0.00112618]
 [ 0.00049649 -0.00107646]
 [ 0.00049617 -0.00106842]
 [-0.00052101  0.00113927]
 [ 0.00049952 -0.0011156 ]
 [-0.0005098   0.0011521 ]
 [-0.00051012  0.001157  ]
 [ 0.00047637 -0.0011079 ]]
(16, 2)


### Calculate the first-layer gradients

Finally 
$$\boxed{dW_1 = dZ_1^T X}$$

$$\boxed{db_1 = \sum_{i=1}^N dZ_{1,i}}$$

In [112]:
dW1 = dZ1.T @ X
db1 = np.sum(dZ1, axis=0)

print(dW1)
print(dW1.shape)

[[-2.28815704e-05  3.46333320e-05 -4.88043940e-06 -2.30740408e-06]
 [ 4.90991964e-05 -7.56276817e-05  1.07552833e-05  5.29953931e-06]]
(2, 4)


Now we have the gradients for all four trainable parameters $W_1, b_1, W_2, b_2$

### Implement the complete training loop

In [113]:

import numpy as np

# 1. Initialize the network parameters
np.random.seed(42)

W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros(2)

W2 = np.random.randn(1, 2) * 0.1
b2 = np.zeros(1)

# 2. Define sigmoid and loss
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)

    return -np.mean(
        y_true * np.log(y_pred)
        + (1 - y_true) * np.log(1 - y_pred)
    )

# 3. Training settings
learning_rate = 0.1
epochs = 10000
loss_history = []

# 4. Train the network
for epoch in range(epochs):

    # Forward pass
    Z1 = X @ W1.T + b1
    A1 = sigmoid(Z1)

    Z2 = A1 @ W2.T + b2
    y_pred = sigmoid(Z2)

    # Calculate loss
    y_true = y.reshape(-1, 1)
    loss = binary_cross_entropy(y_true, y_pred)
    loss_history.append(loss)

    # Backpropagation: output layer
    dZ2 = (y_pred - y_true) / len(y)

    dW2 = dZ2.T @ A1
    db2 = np.sum(dZ2, axis=0)

    # Backpropagation: hidden layer
    dA1 = dZ2 @ W2
    dZ1 = dA1 * A1 * (1 - A1)

    dW1 = dZ1.T @ X
    db1 = np.sum(dZ1, axis=0)

    # Gradient descent
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    # Print progress
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss = {loss:.6f}")

print(f"Final loss: {loss_history[-1]:.6f}")


Epoch     0 | Loss = 0.693149
Epoch  1000 | Loss = 0.693148
Epoch  2000 | Loss = 0.693148
Epoch  3000 | Loss = 0.693148
Epoch  4000 | Loss = 0.693148
Epoch  5000 | Loss = 0.693148
Epoch  6000 | Loss = 0.693147
Epoch  7000 | Loss = 0.693147
Epoch  8000 | Loss = 0.693147
Epoch  9000 | Loss = 0.693147
Final loss: 0.693147


The loss should generally decrease from the initial value, although it may not decrease smoothly at every iteration.

If it doesn't decrease, try investing the gradients, parameter updates, and dataset shapes.

### Test what the network has learned
After training the network, we can test it.

In [114]:

# Make fresh predictions using the trained parameters
Z1 = X @ W1.T + b1
A1 = sigmoid(Z1)
Z2 = A1 @ W2.T + b2
y_pred = sigmoid(Z2)

predictions = (y_pred.flatten() >= 0.5).astype(int)

print("Input matrix       True label   Probability   Prediction")

for i in range(len(X)):
    print(
        X[i],
        "      ",
        y[i],
        "        ",
        round(float(y_pred[i, 0]), 4),
        "       ",
        predictions[i]
    )

accuracy = np.mean(predictions == y)

print("\nTraining accuracy:", accuracy)

Input matrix       True label   Probability   Prediction
[-1 -1 -1 -1]        0          0.5001         1
[-1 -1 -1  1]        1          0.4998         0
[-1 -1  1 -1]        1          0.5006         1
[-1 -1  1  1]        0          0.5003         1
[-1  1 -1 -1]        1          0.5001         1
[-1  1 -1  1]        0          0.4997         0
[-1  1  1 -1]        0          0.5006         1
[-1  1  1  1]        1          0.5002         1
[ 1 -1 -1 -1]        1          0.4998         0
[ 1 -1 -1  1]        0          0.4994         0
[ 1 -1  1 -1]        0          0.5003         1
[ 1 -1  1  1]        1          0.4999         0
[ 1  1 -1 -1]        0          0.4997         0
[ 1  1 -1  1]        1          0.4994         0
[ 1  1  1 -1]        1          0.5002         1
[1 1 1 1]        0          0.4999         0

Training accuracy: 0.5


In [115]:
import numpy as np

np.random.seed(42)

# Network: 4 inputs -> 8 hidden neurons -> 1 output
W1 = np.random.randn(8, 4) * 0.5
b1 = np.zeros(8)

W2 = np.random.randn(1, 8) * 0.5
b2 = np.zeros(1)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(
        y_true * np.log(y_pred)
        + (1 - y_true) * np.log(1 - y_pred)
    )

y_true = y.reshape(-1, 1)
learning_rate = 0.1
epochs = 20000
loss_history = []

for epoch in range(epochs):

    # Forward pass
    Z1 = X @ W1.T + b1
    A1 = sigmoid(Z1)

    Z2 = A1 @ W2.T + b2
    y_pred = sigmoid(Z2)

    loss = binary_cross_entropy(y_true, y_pred)
    loss_history.append(loss)

    # Backpropagation: output layer
    dZ2 = (y_pred - y_true) / len(y)

    dW2 = dZ2.T @ A1
    db2 = np.sum(dZ2, axis=0)

    # Backpropagation: hidden layer
    dA1 = dZ2 @ W2
    dZ1 = dA1 * A1 * (1 - A1)

    dW1 = dZ1.T @ X
    db1 = np.sum(dZ1, axis=0)

    # Gradient descent
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:5d} | Loss = {loss:.6f}")

print("Final loss:", loss_history[-1])


Epoch     0 | Loss = 0.840282
Epoch  2000 | Loss = 0.693076
Epoch  4000 | Loss = 0.692724
Epoch  6000 | Loss = 0.680750
Epoch  8000 | Loss = 0.414978
Epoch 10000 | Loss = 0.250606
Epoch 12000 | Loss = 0.165480
Epoch 14000 | Loss = 0.084796
Epoch 16000 | Loss = 0.041927
Epoch 18000 | Loss = 0.026775
Final loss: 0.01954883866949265


In [116]:
Z1 = X @ W1.T + b1
A1 = sigmoid(Z1)
Z2 = A1 @ W2.T + b2

y_pred = sigmoid(Z2).flatten()
predictions = (y_pred >= 0.5).astype(int)

print("True labels:     ", y)
print("Predictions:     ", predictions)
print("Probabilities:   ", np.round(y_pred, 3))
print("Training accuracy:", np.mean(predictions == y))

True labels:      [0 1 1 0 1 0 0 1 1 0 0 1 0 1 1 0]
Predictions:      [0 1 1 0 1 0 0 1 1 0 0 1 0 1 1 0]
Probabilities:    [0.004 0.995 0.999 0.001 0.941 0.105 0.003 0.943 0.995 0.013 0.026 0.986
 0.003 0.996 0.999 0.001]
Training accuracy: 1.0
